# 02 · Visualización territorial y clustering — EH2025

Este notebook **reproduce visualizaciones a partir de tablas agregadas** producidas por `src/visualize.py` y `src/clustering.py`. El mapa utiliza las geometrías que descargó `src/map_bolivia.py`. No vuelve a ajustar K-Means por defecto, para preservar el experimento reportado.


In [ ]:
from pathlib import Path

def project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "features.py").exists() and (candidate / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError("Abre VS Code en la raíz del proyecto EH2025 o ejecuta el notebook desde esa carpeta.")

ROOT = project_root()
FIG = ROOT / "outputs" / "figures"
MODELS = ROOT / "outputs" / "models"
print("Proyecto:", ROOT)

In [ ]:
import json
import pandas as pd
import plotly.express as px

archivos = {
    "departamentos": FIG / "resumen_departamentos.csv",
    "areas": FIG / "resumen_areas.csv",
    "edades": FIG / "resumen_edades.csv",
    "clusters": FIG / "resumen_clusters.csv",
}
for nombre, ruta in archivos.items():
    assert ruta.exists(), f"Falta {ruta}; ejecuta los scripts en src/"
tablas = {nombre: pd.read_csv(ruta) for nombre, ruta in archivos.items()}
for nombre, tabla in tablas.items():
    print(nombre, list(tabla.columns), "filas:", len(tabla))

## 1. Comparación territorial ponderada

In [ ]:
depto = tablas["departamentos"].sort_values("tasa", ascending=True)
display(depto)
px.bar(depto, x="tasa", y="departamento", orientation="h",
       title="Tasa ponderada de inasistencia por departamento",
       labels={"tasa":"Porcentaje (%)","departamento":"Departamento"}).show()

In [ ]:
areas = tablas["areas"]
display(areas)
px.bar(areas, x=areas.columns[0], y="tasa", text="tasa",
       title="Contraste de inasistencia urbana y rural",
       labels={"tasa":"Porcentaje ponderado (%)"}).show()

## 2. Distribución por edad

In [ ]:
edades = tablas["edades"]
display(edades)
px.line(edades, x="edad", y="tasa", markers=True,
        title="Inasistencia escolar según edad",
        labels={"edad":"Edad (años)","tasa":"Porcentaje ponderado (%)"}).show()

## 3. Mapa coroplético de Bolivia

La capa geográfica procede del servicio público de YPFB, filtrado a los nueve departamentos. El color representa tasas ponderadas; los campos `muestra` y `casos` corresponden a recuentos muestrales sin ponderar.

In [ ]:
import unicodedata

def normalizar(x):
    t = unicodedata.normalize("NFKD", str(x))
    return " ".join("".join(c for c in t if not unicodedata.combining(c)).upper().split())

ruta = ROOT / "data" / "metadata" / "bolivia_departamentos.geojson"
assert ruta.exists(), "Primero ejecuta python src/map_bolivia.py"
geo = json.loads(ruta.read_text(encoding="utf-8"))
assert len(geo["features"]) == 9
for f in geo["features"]:
    f["id"] = normalizar(f["properties"]["departamento"])
map_df = depto.copy()
map_df["codigo_geo"] = map_df["departamento"].map(normalizar)
assert set(map_df["codigo_geo"]) == {f["id"] for f in geo["features"]}
fig = px.choropleth_map(map_df, geojson=geo, locations="codigo_geo", featureidkey="id",
                         color="tasa", hover_name="departamento", color_continuous_scale="YlOrRd",
                         center={"lat":-16.7,"lon":-64.7},zoom=4.6, map_style="carto-positron",
                         title="Bolivia: inasistencia escolar por departamento",
                         labels={"tasa":"Inasistencia (%)"})
fig.update_layout(height=620)
fig.show()

## 4. Segmentación K-Means

In [ ]:
clusters = tablas["clusters"]
display(clusters)
fig = px.scatter(clusters, x="edad_promedio", y="ingreso_mediano", size="muestra",
                 color="tasa_inasistencia", text="cluster", color_continuous_scale="Viridis",
                 hover_data=["porcentaje_rural", "casos"],
                 title="Perfiles K-Means: edad, ingreso e inasistencia",
                 labels={"edad_promedio":"Edad promedio", "ingreso_mediano":"Ingreso mediano",
                         "tasa_inasistencia":"Inasistencia (%)"})
fig.update_traces(textposition="top center")
fig.show()

**Nota metodológica:** K=2 tuvo un Silhouette Score de 0,4528. Los clusters se distinguen principalmente por la variable urbana/rural (0% y 99,74% rurales). No representan grupos sociales independientes del territorio; son perfiles exploratorios, no efectos causales.

## 5. Archivos interactivos para entrega

Se conservan las versiones HTML en `outputs/figures/01_...html` a `05_...html`. Estas figuras se integran en `docs/reporte_final.qmd` y requieren preservar sus rutas relativas al distribuir el informe.